Create a dynamic data dashboard that visualizes classification results and key metrics for fake news detection.

In [5]:

%%writefile app.py

import pandas as pd
import streamlit as st
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


st.set_page_config(
    page_title="VeriFact Dashboard",
    page_icon="📰",
    layout="wide"
)

# Dashboard styling
st.markdown("""
<style>
.block-container {
    padding-top: 2rem;
    padding-bottom: 2rem;
    max-width: 1400px;
}

h1 {
    font-size: 36px !important;
}

h2, h3 {
    margin-top: 1rem !important;
}

[data-testid="stMetric"] {
    background-color: #172338;
    padding: 20px;
    border-radius: 12px;
    border: 1px solid #263954;
}

[data-testid="stMetricLabel"] {
    font-size: 15px !important;
}

[data-testid="stMetricValue"] {
    font-size: 28px !important;
}

div[data-testid="stPlotlyChart"] {
    background-color: #111827;
    border-radius: 12px;
    padding: 10px;
}

.stSelectbox {
    margin-bottom: 10px;
}
</style>
""", unsafe_allow_html=True)


# Load datasets
real = pd.read_csv("True.csv")
fake = pd.read_csv("Fake.csv")
external = pd.read_csv("external_sources.csv")


# Add labels
real["label"] = 1
fake["label"] = 0

data = pd.concat([real, fake], ignore_index=True)

# Prepare news text
data["news_text"] = data["title"].fillna("") + " " + data["text"].fillna("")
external["news_text"] = external["title"].fillna("") + " " + external["text"].fillna("")

data = data[data["news_text"].str.strip() != ""]
external = external[external["news_text"].str.strip() != ""]


# Split data
X_train, X_test, y_train, y_test = train_test_split(
    data["news_text"],
    data["label"],
    test_size=0.2,
    random_state=42,
    stratify=data["label"]
)


# TF-IDF
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)


# Evaluate model
y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)


# Predict external news
external_tfidf = tfidf.transform(external["news_text"])

external["prediction"] = model.predict(external_tfidf)
external["confidence"] = model.predict_proba(external_tfidf).max(axis=1)

external["predicted_news"] = external["prediction"].map({
    1: "Real",
    0: "Fake"
})


# Dashboard heading
st.title("📰 VeriFact")
st.markdown("### Fake News Detection Dashboard")
st.caption("Classification performance and external news analysis")

st.divider()


# Model performance
st.subheader("Model Performance")

col1, col2, col3, col4 = st.columns(4)

col1.metric("Accuracy", f"{accuracy:.2%}")
col2.metric("Precision", f"{precision:.2%}")
col3.metric("Recall", f"{recall:.2%}")
col4.metric("F1 Score", f"{f1:.2%}")


st.divider()


# External news metrics
st.subheader("External News Classification")

total = len(external)
fake_count = (external["prediction"] == 0).sum()
real_count = (external["prediction"] == 1).sum()

col1, col2, col3 = st.columns(3)

col1.metric("Total Articles", f"{total:,}")
col2.metric("Predicted Fake", f"{fake_count:,}")
col3.metric("Predicted Real", f"{real_count:,}")


st.divider()


# Charts
st.subheader("News Classification Analysis")

col1, col2 = st.columns(2)

with col1:
    prediction_counts = external["predicted_news"].value_counts().reset_index()
    prediction_counts.columns = ["Category", "Count"]

    fig1 = px.pie(
        prediction_counts,
        names="Category",
        values="Count",
        hole=0.5,
        title="Real vs Fake News Distribution"
    )

    fig1.update_layout(
        height=420,
        template="plotly_dark",
        legend_title="News Category",
        margin=dict(t=60, b=30, l=20, r=20)
    )

    st.plotly_chart(fig1, use_container_width=True)


with col2:
    type_counts = external.groupby(
        ["type", "predicted_news"]
    ).size().reset_index(name="Count")

    fig2 = px.bar(
        type_counts,
        x="type",
        y="Count",
        color="predicted_news",
        barmode="group",
        title="Predictions by News Type",
        labels={
            "type": "News Type",
            "Count": "Article Count",
            "predicted_news": "Prediction"
        }
    )

    fig2.update_layout(
        height=420,
        template="plotly_dark",
        margin=dict(t=60, b=50, l=30, r=20)
    )

    st.plotly_chart(fig2, use_container_width=True)


st.divider()


# Article explorer
st.subheader("Explore News Articles")
st.caption("Filter articles to explore classification results.")


col1, col2 = st.columns(2)

with col1:
    selected_type = st.selectbox(
        "News Type",
        ["All"] + sorted(external["type"].dropna().unique().tolist())
    )

with col2:
    selected_prediction = st.selectbox(
        "Prediction",
        ["All", "Fake", "Real"]
    )


# Apply filters
filtered_data = external.copy()

if selected_type != "All":
    filtered_data = filtered_data[filtered_data["type"] == selected_type]

if selected_prediction != "All":
    filtered_data = filtered_data[
        filtered_data["predicted_news"] == selected_prediction
    ]


st.metric("Matching Articles", f"{len(filtered_data):,}")


# Display table
st.dataframe(
    filtered_data[[
        "title",
        "type",
        "predicted_news",
        "confidence"
    ]].rename(columns={
        "title": "News Article",
        "type": "News Type",
        "predicted_news": "Prediction",
        "confidence": "Confidence"
    }),
    use_container_width=True,
    height=400
)


# Download button
st.download_button(
    label="⬇ Download Prediction Results",
    data=external.to_csv(index=False),
    file_name="fake_news_predictions.csv",
    mime="text/csv"
)


Overwriting app.py
